In [41]:
import os, re, glob
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import kagglehub
from huggingface_hub import HfApi, hf_hub_download
from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/machinelearning/DeTai5_HanoiApt"
REPO_DIR = "/content/hanoi_real_estate_prediction"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Nguồn Kaggle 2024
kaggle_path = kagglehub.dataset_download("huytrngquc/hanoi-housing-price-2024")
df1_raw = pd.read_csv(glob.glob(f"{kaggle_path}/*.csv")[0])

# 2. Nguồn GitHub Alonhadat 2025
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/hsgsstudent/hanoi_real_estate_prediction.git {REPO_DIR}
df2_raw = pd.read_json(f"{REPO_DIR}/data/output.json")

# 3. Nguồn Hugging Face Vietnam Real Estates (lọc sẵn Hà Nội & Căn hộ chung cư qua PyArrow)
HF_REPO = "tinixai/vietnam-real-estates"
hf_cols = ["name", "description", "property_type_name", "province_name", "district_name",
           "ward_name", "street_name", "project_name", "price", "area",
           "bedroom_count", "bathroom_count", "published_at"]

hf_files = [f for f in HfApi().list_repo_files(HF_REPO, repo_type="dataset") if f.endswith(".parquet")]
parts = [
    pq.read_table(
        hf_hub_download(HF_REPO, f, repo_type="dataset"),
        columns=hf_cols,
        filters=[("province_name", "=", "Hà Nội"), ("property_type_name", "=", "Căn hộ chung cư")]
    ).to_pandas()
    for f in hf_files
]
df3_raw = pd.concat(parts, ignore_index=True)

print(f"Đã tải xong 3 nguồn | Kaggle: {len(df1_raw):,} | GitHub: {len(df2_raw):,} | HuggingFace: {len(df3_raw):,}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using Colab cache for faster access to the 'hanoi-housing-price-2024' dataset.
Đã tải xong 3 nguồn | Kaggle: 13,535 | GitHub: 5,048 | HuggingFace: 303,589


In [42]:
def parse_area(val):
    if pd.isna(val): return np.nan
    m = re.search(r"(\d+(?:\.\d+)?)", str(val).lower().replace(",", "."))
    return float(m.group(1)) if m else np.nan

def parse_price(val, area):
    s = str(val).lower().strip().replace(",", ".")
    if pd.isna(val) or "thỏa thuận" in s or "thoả thuận" in s: return np.nan
    m = re.search(r"(\d+(?:\.\d+)?)", s)
    if not m: return np.nan
    num = float(m.group(1))
    if "/m" in s or "m2" in s or "m²" in s: return num * 1000 if "tỷ" in s else num
    if "tỷ" in s: return (num * 1000) / area if area > 0 else np.nan
    if "triệu" in s: return num / area if area > 0 else np.nan
    return np.nan

def extract_int(val):
    if pd.isna(val) or str(val).strip() == "0": return np.nan
    m = re.search(r"(\d+)", str(val))
    return float(m.group(1)) if m else np.nan

# --- Xử lý Nguồn 1 (Kaggle 2024) ---
m1 = (df1_raw["PostType"].astype(str).str.contains("chung cư|căn hộ", case=False, na=False) |
      df1_raw["Title"].astype(str).str.contains("chung cư|căn hộ", case=False, na=False))
d1 = df1_raw[m1].copy()
d1["area_m2"] = d1["Area"].apply(parse_area)
d1["price_per_m2"] = d1.apply(lambda r: parse_price(r["Price"], r["area_m2"]), axis=1)
d1["bedrooms"] = d1["Bedrooms"].apply(extract_int)
d1["bathrooms"] = d1["Bathrooms"].apply(extract_int)
d1["has_red_book"] = d1["Legal"].astype(str).str.contains("sổ đỏ|sổ hồng|lâu dài", case=False, na=False).astype(int)
d1["interior_level"] = d1["Interior"].apply(
    lambda x: 2 if any(k in str(x).lower() for k in ["đầy đủ", "full"]) else (1 if "cơ bản" in str(x).lower() else 0)
)
d1["collected_at"] = pd.to_datetime(d1["PostingDate"], dayfirst=True, errors="coerce")
d1["source"] = "kaggle_batdongsan_2024"

# --- Xử lý Nguồn 2 (GitHub Alonhadat 2025) ---
m2 = (df2_raw["type"].astype(str).str.contains("chung cư|căn hộ|tập thể", case=False, na=False) |
      df2_raw["description"].astype(str).str.contains("chung cư|căn hộ", case=False, na=False))
d2 = df2_raw[m2].copy()
d2["area_m2"] = d2["area"].apply(parse_area)
d2["price_per_m2"] = d2.apply(lambda r: parse_price(r["price"], r["area_m2"]), axis=1)
d2["bedrooms"] = d2["bedroom_number"].apply(extract_int)
d2["bathrooms"] = np.nan
d2["has_red_book"] = d2["law"].astype(str).str.contains("sổ", case=False, na=False).astype(int)
d2["interior_level"] = d2["description"].astype(str).str.contains("full nội thất|đầy đủ nội thất", case=False, na=False).astype(int) * 2
d2["collected_at"] = pd.to_datetime(d2["start_date"].astype(str).str.extract(r"(\d{2}/\d{2}/\d{4})")[0], dayfirst=True, errors="coerce")
d2["District"] = d2["address"].apply(lambda a: m.group(1).strip() if (m := re.search(r"(?:Quận|Huyện)\s+([^,]+)", str(a), re.I)) else "Unknown")
d2["Address"] = d2["address"].astype(str)
d2["Title"] = d2["description"].astype(str).str.slice(0, 120)
d2["source"] = "github_alonhadat_2025"

# --- Xử lý Nguồn 3 (Hugging Face) ---
d3 = df3_raw.copy()
d3["price"] = pd.to_numeric(d3["price"], errors="coerce")
d3["area_m2"] = pd.to_numeric(d3["area"], errors="coerce")
d3["price_per_m2"] = d3["price"] / d3["area_m2"] / 1e6
d3["Title"] = d3["name"].fillna(d3["description"]).astype(str).str.slice(0, 120)
d3["Address"] = d3["street_name"].fillna("").astype(str) + ", " + d3["ward_name"].fillna("").astype(str)
d3["District"] = d3["district_name"].fillna("Unknown")
d3["collected_at"] = pd.to_datetime(d3["published_at"], errors="coerce")
d3["bedrooms"] = pd.to_numeric(d3["bedroom_count"], errors="coerce")
d3["bathrooms"] = pd.to_numeric(d3["bathroom_count"], errors="coerce")
d3["has_red_book"] = 1
d3["interior_level"] = d3["description"].astype(str).str.contains("full nội thất|đầy đủ nội thất", case=False, na=False).astype(int) * 2
d3["source"] = "huggingface_vietnam_real_estates"

# --- Gộp, lọc nhiễu & khử trùng lặp chéo ---
cols = ["Title", "Address", "District", "collected_at", "source",
        "area_m2", "price_per_m2", "bedrooms", "bathrooms", "has_red_book", "interior_level"]

df_all = pd.concat([d1[cols], d2[cols], d3[cols]], ignore_index=True)
df_all = df_all[
    df_all["area_m2"].between(15, 500) &
    df_all["price_per_m2"].between(15, 300)
].dropna(subset=["collected_at"]).copy()

df_all["area_round"] = df_all["area_m2"].round(1)
df_all["price_round"] = df_all["price_per_m2"].round(1)
df_final = (
    df_all.drop_duplicates(subset=["District", "area_round", "price_round", "Title"])
          .drop(columns=["area_round", "price_round"])
          .reset_index(drop=True)
)
df_final["log_price_m2"] = np.log(df_final["price_per_m2"])

out_path = f"{SAVE_DIR}/hanoi_apt_clean_5k.csv"
df_final.to_csv(out_path, index=False)

print(f"Thu thập thô chung cư 3 nguồn : {len(d1) + len(d2) + len(d3):,} tin")
print(f"Tổng bản ghi sạch cuối cùng    : {len(df_final):,} tin")
print(f"Khoảng thời gian dữ liệu       : {df_final['collected_at'].min().date()} -> {df_final['collected_at'].max().date()}")
print(df_final["source"].value_counts())
display(df_final.head(3))

Thu thập thô chung cư 3 nguồn : 309,133 tin
Tổng bản ghi sạch cuối cùng    : 189,293 tin
Khoảng thời gian dữ liệu       : 2024-11-18 -> 2026-03-30
source
huggingface_vietnam_real_estates    185673
kaggle_batdongsan_2024                3123
github_alonhadat_2025                  497
Name: count, dtype: int64


,Title,Address,District,collected_at,source,area_m2,price_per_m2,bedrooms,bathrooms,has_red_book,interior_level,log_price_m2
0,"Quỹ căn hộ Studio, 1N, 2N, 3N chính chủ gửi bá...","Dự án Vinhomes Ocean Park Gia Lâm, Xã Dương Xá...",Gia Lâm,2024-12-11,kaggle_batdongsan_2024,43.00,51.162791,1.0,1.0,1,2,3.935013
1,Quỹ căn đặc biệt dự án Viha Complex - Viha Lec...,"Viha Complex, 107, Đường Nguyễn Tuân, Phường T...",Thanh Xuân,2024-12-16,kaggle_batdongsan_2024,72.11,83.000000,2.0,NaN,0,0,4.418841
2,"Quỹ căn stu,1-3PN chính chủ view đẹp, pháp lý ...","Dự án Vinhomes Ocean Park Gia Lâm, Xã Dương Xá...",Gia Lâm,2024-12-05,kaggle_batdongsan_2024,59.00,44.915254,2.0,1.0,1,0,3.804777


In [43]:
# Lấy toàn bộ tin 2024-2025 từ Kaggle & GitHub + lấy mẫu ngẫu nhiên có phân tầng theo tuần từ HuggingFace
df_recent = df_final[df_final["collected_at"] >= "2025-11-01"].copy()
df_recent.to_csv(f"{SAVE_DIR}/hanoi_apt_recent_sub.csv", index=False)
print(f"Tập dữ liệu giai đoạn gần nhất (từ 11/2025): {len(df_recent):,} bản ghi")

Tập dữ liệu giai đoạn gần nhất (từ 11/2025): 84,838 bản ghi


In [44]:
import os, shutil
from google.colab import drive

drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/machinelearning/DeTai5_HanoiApt"

folders = [
    "data/raw", "data/processed", "src", "notebooks",
    "artifacts/tables", "artifacts/figures", "docs"
]
for f in folders:
    os.makedirs(os.path.join(BASE_DIR, f), exist_ok=True)

# Tạo file __init__.py để import src dễ dàng
open(os.path.join(BASE_DIR, "src/__init__.py"), "a").close()

# Di chuyển 2 file dữ liệu hiện có vào đúng thư mục data/
moves = {
    "raw_hanoi_housing.csv": "data/raw/raw_hanoi_housing.csv",
    "hanoi_apt_clean_5k.csv": "data/processed/hanoi_apt_clean_5k.csv",
}
for src_name, dst_rel in moves.items():
    src_path = os.path.join(BASE_DIR, src_name)
    dst_path = os.path.join(BASE_DIR, dst_rel)
    if os.path.exists(src_path) and not os.path.exists(dst_path):
        shutil.move(src_path, dst_path)
        print(f"Đã chuyển: {src_name} -> {dst_rel}")

print("\nHoàn tất khởi tạo cấu trúc dự án tại:", BASE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã chuyển: raw_hanoi_housing.csv -> data/raw/raw_hanoi_housing.csv
Đã chuyển: hanoi_apt_clean_5k.csv -> data/processed/hanoi_apt_clean_5k.csv

Hoàn tất khởi tạo cấu trúc dự án tại: /content/drive/MyDrive/machinelearning/DeTai5_HanoiApt
